<a href="https://colab.research.google.com/github/Baelfyre/A3101-ml-linear-algebra-project-Team-3/blob/main/MO_IT162_Milestone_1_ML_Solution_EDA_BSIT_A3101_G_Lugo%2C_J_Ongo%2C_K_Ponteres%2C_M_Goyon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MO-IT162 Milestone 1: ML Solution EDA

**Project:** FinMark Corporation  
**Course:** MO-IT162 Math for Machine Learning: Linear Algebra  
**Section:** A3101  
**Milestone:** Milestone 1 - ML Solution EDA  
**Team:** Team 3  

## Team Members

- Ghaylord Benedict Lugod
- James Lynelle Ongo
- Kaselyn Cates Ponteres
- Mark Anthony Goyon

**Environment:** Google Colab / Jupyter Notebook  
**Libraries:** Pandas, NumPy, Matplotlib, Seaborn

## Project Overview

This notebook documents the Exploratory Data Analysis (EDA) of FinMark Corporation's finalized preprocessed datasets from Week 3.

The analysis focuses on three areas of customer data:

- customer demographics;
- customer transactions; and
- social media interactions.

The purpose of the EDA is to examine the structure and quality of the datasets, summarize their statistical characteristics, identify patterns and anomalies, explore relationships between relevant variables, and assess whether the data is suitable for succeeding visualization and machine learning activities.

The analysis follows a structured process from data verification and preprocessing review to statistical analysis, interpretation, and machine learning readiness assessment.

In [2]:
# @title
# =============================================================================
# LOAD LIBRARIES
# =============================================================================
# The following libraries are used throughout the Exploratory Data Analysis.
#
# Pandas:
# Used for loading, inspecting, grouping, cleaning, and summarizing datasets.
#
# NumPy:
# Used for numerical and statistical operations.
#
# Matplotlib:
# Used to create exploratory charts and figures.
#
# Seaborn:
# Used for statistical visualizations such as histograms, box plots,
# count plots, scatter plots, and heatmaps.
#
# Google Colab normally includes these libraries by default, so installation
# is generally not required.
#
# If a library is missing in another environment, the following installation
# command may be used:
#
# !pip install pandas numpy matplotlib seaborn

import pandas as pd

import numpy as np

import matplotlib.pyplot as plt

import seaborn as sns


# -----------------------------------------------------------------------------
# CHECK LIBRARY VERSIONS
# -----------------------------------------------------------------------------
# Recording the versions helps document the environment used for the analysis.

print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)
print("Matplotlib version:", plt.matplotlib.__version__)
print("Seaborn version:", sns.__version__)


# -----------------------------------------------------------------------------
# NOTEBOOK DISPLAY SETTINGS
# -----------------------------------------------------------------------------
# Display all DataFrame columns when reviewing the FinMark datasets.

pd.set_option("display.max_columns", None)


# Set a readable default figure size for exploratory visualizations.

plt.rcParams["figure.figsize"] = (9, 5)

Pandas version: 2.2.3
NumPy version: 2.1.3
Matplotlib version: 3.10.0
Seaborn version: 0.13.2


# 1. Business Problem and Analytical Needs

## 1.1 FinMark Business Problem

FinMark Corporation needs to improve how it analyzes customer information and generates useful insights from customer demographics, transaction activity, and social media interactions.

Before the available data can support future machine learning activities, the datasets must first be examined to determine whether they are reliable, consistent, and suitable for analysis.

The Exploratory Data Analysis will therefore investigate the statistical characteristics, distributions, patterns, relationships, and anomalies present in the available FinMark data.

## 1.2 Stakeholders and User Needs

The primary stakeholders include:

- FinMark decision-makers;
- FinMark data analysts; and
- the future machine learning development team.

The analysis should help these stakeholders:

- understand the quality and structure of the available data;
- identify important customer and transaction characteristics;
- detect possible data-quality risks and unusual observations;
- identify patterns or relationships that may require deeper investigation; and
- prepare reliable analytical data for Milestone 2 and future machine learning activities.

## 1.3 Objectives

The analysis aims to:

1. verify the structure and quality of the finalized Week 3 datasets;
2. review the preprocessing procedures applied during Weeks 2 and 3;
3. summarize numerical and categorical variables using appropriate statistical methods;
4. examine distributions, frequencies, variability, and potential outliers;
5. compare relevant customer, transaction, and social media groups;
6. investigate patterns, trends, and relationships between relevant variables;
7. interpret findings in the context of FinMark's analytical needs; and
8. assess considerations that should be carried forward into future visualization and machine learning activities.

## 1.4 Scope

### Included

- Customer demographics
- Customer transactions
- Social media interactions
- Data verification
- Preprocessing review
- Descriptive statistics
- Frequency and distribution analysis
- Grouped analysis
- Temporal analysis
- Exploratory visualizations
- Cross-dataset customer analysis
- Relationship analysis
- Machine learning readiness assessment

### Outside the Scope

- Machine learning model training
- Model prediction
- Model performance evaluation
- Production deployment
- Unsupported causal conclusions

The findings in this milestone describe patterns and associations observed within the supplied FinMark datasets. Relationships identified during EDA should not automatically be interpreted as causal.

## Notebook Display Standards

To keep numerical outputs consistent and readable throughout the analysis:

- **Raw discrete values and counts** are displayed as whole numbers.
- **Calculated statistics** such as mean, variance, and standard deviation are displayed to 2 decimal places.
- **Transaction amounts** are displayed to 2 decimal places.
- **Percentages** are displayed to 2 decimal places with `%`.
- **Correlations** are displayed to 2 decimal places.
- **Dates** use the `YYYY-MM-DD` format.

Display formatting does not change the underlying dataset values or calculation precision.

In [3]:
# @title
# Standard display helpers used throughout the notebook.
# These affect presentation only and do not modify the underlying dataset.

def fmt_whole(value):
    """Display counts and discrete values as whole numbers."""
    return f"{int(value):,}"


def fmt_stat(value):
    """Display calculated statistics to 2 decimal places."""
    return f"{value:,.2f}"


def fmt_amount(value):
    """Display monetary values to 2 decimal places."""
    return f"{value:,.2f}"


def fmt_percent(value):
    """Display percentages to 2 decimal places."""
    return f"{value:.2f}%"


def fmt_flexible(value):
    """Use a whole number when exact, otherwise use 2 decimal places."""
    if float(value).is_integer():
        return f"{int(value):,}"
    return f"{value:,.2f}"


def show_describe(series, value_type="numeric"):
    """
    Display Pandas describe() results using standardized formatting.

    value_type:
        "discrete" -> age-like variables
        "amount"   -> monetary variables
        "numeric"  -> general numerical variables
    """

    summary = series.describe()

    formatted = {}

    for statistic, value in summary.items():

        # Record counts are always whole numbers.
        if statistic == "count":
            formatted[statistic] = fmt_whole(value)

        # Monetary statistics always use 2 decimal places.
        elif value_type == "amount":
            formatted[statistic] = fmt_amount(value)

        # For discrete variables such as age:
        # mean and standard deviation remain calculated statistics,
        # while min, quartiles, median, and max use flexible formatting.
        elif value_type == "discrete":
            if statistic in ["mean", "std"]:
                formatted[statistic] = fmt_stat(value)
            else:
                formatted[statistic] = fmt_flexible(value)

        # General calculated numerical values use 2 decimal places.
        else:
            formatted[statistic] = fmt_stat(value)

    display(
        pd.DataFrame.from_dict(
            formatted,
            orient="index",
            columns=[series.name]
        )
    )

# 2. Data Collection and Dataset Overview

The analysis uses the three datasets provided for the FinMark Corporation project.

These datasets represent different aspects of customer behavior and are used together to provide a more complete view of FinMark's customers.

## 2.1 Customer Demographics

The Customer Demographics dataset contains customer-level descriptive information.

Key variables include:

- CustomerID
- Age
- Gender
- IncomeLevel
- SignupDate

This dataset provides information that can be used to describe and compare different customer groups.

## 2.2 Customer Transactions

The Customer Transactions dataset contains records of customer purchasing activity.

Key variables include:

- TransactionID
- CustomerID
- TransactionDate
- Amount
- ProductCategory
- PaymentMethod

This dataset is used to examine transaction behavior, purchasing patterns, product categories, payment methods, and changes in transaction activity over time.

## 2.3 Social Media Interactions

The Social Media Interactions dataset contains customer engagement activity recorded across social media platforms.

Key variables include:

- InteractionID
- CustomerID
- InteractionDate
- Platform
- InteractionType
- Sentiment

This dataset supports analysis of customer engagement, interaction behavior, platform activity, and sentiment patterns.

## 2.4 Dataset Relationships

The three datasets are related through **CustomerID**.

CustomerID allows customer transaction and social media activity to be associated with customer demographic information.

The datasets should not be combined without first considering their level of detail because a single customer may have multiple transactions and multiple social media interactions.

In [4]:
# @title
# =============================================================================
# LOAD FINALIZED WEEK 3 PREPROCESSED DATASETS
# =============================================================================
# The finalized Week 3 datasets are stored in the team's Google Drive project
# folder. Mount Google Drive so Colab can access the submitted cleaned files
# directly without requiring a manual upload each session.

from google.colab import drive

drive.mount("/content/drive")


# -----------------------------------------------------------------------------
# PROJECT DATA PATH
# -----------------------------------------------------------------------------
# This is the folder containing the finalized Week 3 preprocessing submission.

base_path = (
    "/content/drive/MyDrive/"
    "3rd Year 26-27 | Term 1/"
    "MO-IT162 - MMLA/"
    "MS1/"
    "Week3_Preprocessing_Submission"
)


# -----------------------------------------------------------------------------
# FINALIZED CLEANED DATASETS
# -----------------------------------------------------------------------------

demographics_path = (
    f"{base_path}/customer_demographics_cleaned.csv"
)

transactions_path = (
    f"{base_path}/customer_transactions_cleaned.csv"
)

social_path = (
    f"{base_path}/social_media_interactions_cleaned.csv"
)


# -----------------------------------------------------------------------------
# LOAD DATASETS
# -----------------------------------------------------------------------------
# Each dataset is loaded separately so its structure and quality can be
# verified before cross-dataset analysis is performed.

demographics = pd.read_csv(demographics_path)

transactions = pd.read_csv(transactions_path)

social = pd.read_csv(social_path)


# -----------------------------------------------------------------------------
# CONFIRM SUCCESSFUL LOADING
# -----------------------------------------------------------------------------
# Display the number of rows and columns in each cleaned dataset.

print("Customer Demographics:", demographics.shape)

print("Customer Transactions:", transactions.shape)

print("Social Media Interactions:", social.shape)

Mounted at /content/drive
Customer Demographics: (2378, 6)
Customer Transactions: (2420, 6)
Social Media Interactions: (2411, 6)


In [5]:
# @title
# Review the basic structure of each FinMark dataset.
# This checks sample records, dimensions, columns, data types, and unique values.

datasets = {
    "Customer Demographics": demographics,
    "Customer Transactions": transactions,
    "Social Media Interactions": social
}

for name, df in datasets.items():
    print(f"\n{'=' * 60}")
    print(name)
    print("=" * 60)

    # Preview sample records
    print("\nFirst 5 rows:")
    display(df.head())

    # Check dataset dimensions
    print(f"\nRows: {df.shape[0]} | Columns: {df.shape[1]}")

    # Review available variables
    print("\nColumns:")
    print(df.columns.tolist())

    # Check data types and non-null values
    print("\nDataset information:")
    df.info()

    # Count unique values in each variable
    print("\nUnique values per column:")
    display(df.nunique())


Customer Demographics

First 5 rows:


,customer_id,age,gender,location,income_level,signup_date
0,9207fa75-5758-48d1-94ad-19c041e0520f,51,Female,Jensenberg,Low,2022-11-17
1,50118139-7264-428f-81cc-a25fddc5d6dd,44,Male,Port Carl,Medium,2024-06-10
2,7d1f2bbc-8d16-4fbc-9b37-ece3324e8ed4,50,Female,Jessebury,High,2023-08-24
3,2de49c7c-32ae-4ba8-b058-622a090d7094,53,Female,Emilyville,Low,2022-02-13
4,8602d631-457c-49c1-8b59-8efb2a4448d4,51,Male,East Keithville,High,2022-04-17



Rows: 2378 | Columns: 6

Columns:
['customer_id', 'age', 'gender', 'location', 'income_level', 'signup_date']

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2378 entries, 0 to 2377
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   customer_id   2378 non-null   object
 1   age           2378 non-null   int64 
 2   gender        2378 non-null   object
 3   location      2378 non-null   object
 4   income_level  2378 non-null   object
 5   signup_date   2378 non-null   object
dtypes: int64(1), object(5)
memory usage: 111.6+ KB

Unique values per column:


,0
customer_id,2378
age,53
gender,2
location,2185
income_level,3
signup_date,1332



Customer Transactions

First 5 rows:


...

,0
customer_id,1642
transaction_id,2420
transaction_date,701
amount,2342
product_category,5
payment_method,4



Social Media Interactions

First 5 rows:


,customer_id,interaction_id,interaction_date,platform,interaction_type,sentiment
0,08a911a3-65e6-4f5d-a6a1-ae7ddcbe28a2,a83fa04c-f109-4f24-8ce1-2078154f6a1c,2024-05-24,Instagram,Comment,Neutral
1,efdfdfc9-5dbb-4478-911a-101a390a0285,28a69c4b-a2e4-4c74-a130-1132d7733fdf,2023-11-01,Instagram,Like,Neutral
2,3e44871b-f56c-4576-b1ca-d1dc999e2166,0c409883-8396-48e4-83fb-887329848696,2023-12-18,Instagram,Comment,Positive
3,aa5eea4b-c948-41f4-9285-229a470002aa,4034dadf-6541-40d6-a7f0-16b20a009c04,2023-11-15,Instagram,Share,Positive
4,481bcc62-9dcb-4766-bbdd-79f5554d73f8,785b7854-4548-4251-83f2-a25da78bfe40,2024-05-28,Twitter,Like,Positive



Rows: 2411 | Columns: 6

Columns:
['customer_id', 'interaction_id', 'interaction_date', 'platform', 'interaction_type', 'sentiment']

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2411 entries, 0 to 2410
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   customer_id       2411 non-null   object
 1   interaction_id    2411 non-null   object
 2   interaction_date  2411 non-null   object
 3   platform          2411 non-null   object
 4   interaction_type  2411 non-null   object
 5   sentiment         2411 non-null   object
dtypes: object(6)
memory usage: 113.1+ KB

Unique values per column:


,0
customer_id,1640
interaction_id,2411
interaction_date,365
platform,3
interaction_type,3
sentiment,5


# 3. Variable Classification

Before performing statistical analysis, the variables are classified according to their analytical role and level of measurement.

| Dataset | Variable | Type | Measurement / Role |
|---|---|---|---|
| Demographics | CustomerID | Identifier | Nominal key |
| Demographics | Age | Quantitative | Ratio |
| Demographics | Gender | Categorical | Nominal |
| Demographics | IncomeLevel | Categorical | Ordinal |
| Demographics | SignupDate | Temporal | Date |
| Transactions | TransactionID | Identifier | Nominal key |
| Transactions | CustomerID | Identifier | Nominal key |
| Transactions | TransactionDate | Temporal | Date |
| Transactions | Amount | Quantitative | Ratio |
| Transactions | ProductCategory | Categorical | Nominal |
| Transactions | PaymentMethod | Categorical | Nominal |
| Social Media | InteractionID | Identifier | Nominal key |
| Social Media | CustomerID | Identifier | Nominal key |
| Social Media | InteractionDate | Temporal | Date |
| Social Media | Platform | Categorical | Nominal |
| Social Media | InteractionType | Categorical | Nominal |
| Social Media | Sentiment | Categorical | Ordinal* |

> **Note:** Identifier fields such as `CustomerID`, `TransactionID`, and `InteractionID` are used to identify or connect records and are not treated as numerical measurements.  
> *Sentiment is treated as ordinal only if its categories follow a validated order.*

# 4. Data Verification

Before using the finalized datasets for statistical analysis, the data must be verified to confirm that the preprocessing performed during Weeks 2 and 3 produced accurate, consistent, and usable datasets.

The verification process will check the following areas:

1. **Missing values**  
   Determine whether important fields still contain missing observations.

2. **Exact duplicate records**  
   Confirm whether duplicate rows identified during preprocessing were properly addressed.

3. **Identifier integrity**  
   Check whether supposedly unique identifiers such as `TransactionID` and `InteractionID` remain unique.

4. **Data types**  
   Confirm that numerical and temporal variables use appropriate data types.

5. **Date validity**  
   Verify that date values can be interpreted consistently.

6. **Numerical ranges**  
   Review minimum and maximum values to identify remaining invalid or questionable observations.

7. **Categorical consistency**  
   Check category labels for inconsistent spelling, formatting, unexpected values, or missing categories.

8. **Cross-dataset relationships**  
   Verify that `CustomerID` values used in the transaction and social media datasets correspond to customers represented in the demographics dataset.

Questionable observations will be documented before deciding whether additional treatment is necessary.

The next verification steps will therefore follow the sequence:

**Check → Identify Issue → Evaluate → Document → Determine Treatment**

In [6]:
# @title
# Check for remaining missing values in each cleaned dataset.
# Both count and percentage are shown so completeness can be compared.

missing_rows = []

for dataset_name, df in datasets.items():
    for column in df.columns:
        missing_count = df[column].isna().sum()
        missing_percent = (missing_count / len(df)) * 100

        missing_rows.append({
            "Dataset": dataset_name,
            "Column": column,
            "Missing Count": missing_count,
            "Missing %": round(missing_percent, 2)
        })

missing_summary = pd.DataFrame(missing_rows)

display(missing_summary)

print(
    "Total remaining missing values:",
    missing_summary["Missing Count"].sum()
)

,Dataset,Column,Missing Count,Missing %
0,Customer Demographics,customer_id,0,0.0
1,Customer Demographics,age,0,0.0
2,Customer Demographics,gender,0,0.0
3,Customer Demographics,location,0,0.0
4,Customer Demographics,income_level,0,0.0
5,Customer Demographics,signup_date,0,0.0
6,Customer Transactions,customer_id,0,0.0
7,Customer Transactions,transaction_id,0,0.0
8,Customer Transactions,transaction_date,0,0.0
9,Customer Transactions,amount,0,0.0


Total remaining missing values: 0


In [7]:
# @title
# Check exact duplicate rows and duplicated primary identifiers.
# These are reviewed separately because a repeated ID is not always
# the same issue as an exact duplicate row.

id_columns = {
    "Customer Demographics": "customer_id",
    "Customer Transactions": "transaction_id",
    "Social Media Interactions": "interaction_id"
}

duplicate_rows = []

for dataset_name, df in datasets.items():
    id_column = id_columns[dataset_name]

    duplicate_rows.append({
        "Dataset": dataset_name,
        "Rows": len(df),
        "Exact Duplicate Rows": df.duplicated().sum(),
        "Duplicate Primary IDs": df[id_column].duplicated().sum(),
        "Unique Primary IDs": df[id_column].nunique()
    })

duplicate_summary = pd.DataFrame(duplicate_rows)

display(duplicate_summary)

,Dataset,Rows,Exact Duplicate Rows,Duplicate Primary IDs,Unique Primary IDs
0,Customer Demographics,2378,0,0,2378
1,Customer Transactions,2420,0,0,2420
2,Social Media Interactions,2411,0,0,2411


In [8]:
# @title
# Verify the three date variables and convert them to Pandas datetime.
# errors="coerce" changes an unparseable value to NaT so it can be detected.

date_columns = {
    "Customer Demographics": "signup_date",
    "Customer Transactions": "transaction_date",
    "Social Media Interactions": "interaction_date"
}

date_rows = []

for dataset_name, df in datasets.items():
    column = date_columns[dataset_name]

    dtype_before = str(df[column].dtype)

    # Convert the date field for temporal analysis.
    df[column] = pd.to_datetime(
        df[column],
        errors="coerce"
    )

    date_rows.append({
        "Dataset": dataset_name,
        "Date Column": column,
        "Type Before": dtype_before,
        "Type After": str(df[column].dtype),
        "Missing / Unparseable": df[column].isna().sum(),
        "Earliest Date": df[column].min(),
        "Latest Date": df[column].max()
    })

date_summary = pd.DataFrame(date_rows)

display(date_summary)

,Dataset,Date Column,Type Before,Type After,Missing / Unparseable,Earliest Date,Latest Date
0,Customer Demographics,signup_date,object,datetime64[ns],0,2019-07-01,2024-06-30
1,Customer Transactions,transaction_date,object,datetime64[ns],0,2022-07-01,2024-06-30
2,Social Media Interactions,interaction_date,object,datetime64[ns],0,2023-07-01,2024-06-30


In [9]:
# @title
# Review the main numerical variables using standardized display formatting.

print("AGE SUMMARY")
show_describe(
    demographics["age"],
    value_type="discrete"
)

print("\nTRANSACTION AMOUNT SUMMARY")
show_describe(
    transactions["amount"],
    value_type="amount"
)

# Review the main categorical variables.
# This helps identify unexpected labels or unresolved inconsistencies.

categorical_columns = {
    "Demographics": (
        demographics,
        ["gender", "income_level"]
    ),
    "Transactions": (
        transactions,
        ["product_category", "payment_method"]
    ),
    "Social Media": (
        social,
        ["platform", "interaction_type", "sentiment"]
    )
}

for dataset_name, (df, columns) in categorical_columns.items():

    print(f"\n{'=' * 50}")
    print(dataset_name)
    print("=" * 50)

    for column in columns:

        print(f"\n{column}")

        display(
            df[column]
            .value_counts(dropna=False)
            .to_frame("Count")
        )


# location contains many unique values, so summarize it instead
# of printing every individual location.

print("\nLOCATION SUMMARY")
print(
    "Unique locations:",
    demographics["location"].nunique()
)

display(
    demographics["location"]
    .value_counts()
    .head(10)
    .to_frame("Top 10 Count")
)

AGE SUMMARY


,age
count,"2,378"
mean,44.73
std,15.44
min,18
25%,31
50%,45
75%,58
max,70



TRANSACTION AMOUNT SUMMARY


,amount
count,"2,420"
mean,495.25
std,295.71
min,0.00
25%,234.28
50%,502.12
75%,747.75
max,999.86



Demographics

gender


...

,Count
interaction_type,
Comment,808
Share,808
Like,795



sentiment


,Count
sentiment,
Neutral,790
Positive,790
Negative,780
Very Positive,26
Very Negative,25



LOCATION SUMMARY
Unique locations: 2185


,Top 10 Count
location,
Port Michael,5
Michaelville,4
West Robert,4
New James,3
Williamshaven,3
Adamshire,3
East Brittany,3
West Michael,3
Michaelton,3


In [10]:
# @title
# Verify customer relationships across the three datasets.
# customer_id is the shared key linking demographic, transaction,
# and social-media information.

demographic_ids = set(
    demographics["customer_id"].dropna()
)

transaction_ids = set(
    transactions["customer_id"].dropna()
)

social_ids = set(
    social["customer_id"].dropna()
)


relationship_summary = pd.DataFrame({
    "Measure": [
        "Customers in demographics",
        "Customers represented in transactions",
        "Customers represented in social media",
        "Transaction customer IDs not in demographics",
        "Social customer IDs not in demographics",
        "Customers represented in all three datasets"
    ],
    "Count": [
        len(demographic_ids),
        len(transaction_ids),
        len(social_ids),
        len(transaction_ids - demographic_ids),
        len(social_ids - demographic_ids),
        len(
            demographic_ids
            & transaction_ids
            & social_ids
        )
    ]
})

display(relationship_summary)


# Show how much of the demographic customer base is represented
# in the transaction and social-media datasets.

transaction_coverage = (
    len(demographic_ids & transaction_ids)
    / len(demographic_ids)
    * 100
)

social_coverage = (
    len(demographic_ids & social_ids)
    / len(demographic_ids)
    * 100
)

print(
    f"Transaction customer coverage: "
    f"{transaction_coverage:.2f}%"
)

print(
    f"Social-media customer coverage: "
    f"{social_coverage:.2f}%"
)

,Measure,Count
0,Customers in demographics,2378
1,Customers represented in transactions,1642
2,Customers represented in social media,1640
3,Transaction customer IDs not in demographics,336
4,Social customer IDs not in demographics,336
5,Customers represented in all three datasets,718


Transaction customer coverage: 54.92%
Social-media customer coverage: 54.84%


## 4.1 Verification Findings

The finalized Week 3 datasets were verified before beginning statistical analysis.

### Missing Values

**Finding:** No missing values remain in the three cleaned datasets.

**Evidence:** All 18 variables across Customer Demographics, Customer Transactions, and Social Media Interactions recorded `0` missing values.

**Interpretation:** The datasets are complete for the variables retained after preprocessing, so no additional missing-value treatment is required before EDA.

### Duplicate and Identifier Integrity

**Finding:** No exact duplicate records or duplicate primary identifiers remain.

**Evidence:**

- Customer Demographics: 2,378 rows and 2,378 unique `customer_id` values
- Customer Transactions: 2,420 rows and 2,420 unique `transaction_id` values
- Social Media Interactions: 2,411 rows and 2,411 unique `interaction_id` values

All three datasets recorded `0` exact duplicate rows and `0` duplicate primary IDs.

**Interpretation:** Each cleaned record is uniquely represented according to its primary identifier, reducing the risk of duplicate records distorting the analysis.

### Date Validity

**Finding:** All retained date values were successfully parsed into a consistent datetime format.

**Evidence:**

- `signup_date`: July 1, 2019 to June 30, 2024
- `transaction_date`: July 1, 2022 to June 30, 2024
- `interaction_date`: July 1, 2023 to June 30, 2024
- Missing or unparseable dates: `0`

**Interpretation:** The three date variables are suitable for temporal analysis. However, their observation periods differ, which should be considered when comparing activity across datasets.

### Numerical and Categorical Validity

**Finding:** The retained numerical values fall within the accepted cleaned ranges, while the major categorical variables contain a limited and consistent set of labels.

**Evidence:**

- Customer age ranges from 18 to 70 years, with a mean of 44.73 and median of 45.
- Transaction amounts range from 0.00 to 999.86, with a mean of 495.25 and median of 502.12.
- Gender contains 2 categories.
- Income level contains 3 categories.
- Product category contains 5 categories.
- Payment method contains 4 categories.
- Platform contains 3 categories.
- Interaction type contains 3 categories.
- Sentiment contains 5 categories.

The `location` variable contains 2,185 unique values among 2,378 customers, indicating very high cardinality.

**Interpretation:** The principal numerical and categorical variables are suitable for EDA. The high cardinality of `location` makes it less suitable for simple frequency-based analysis without further grouping or aggregation. Sentiment also contains relatively small `Very Positive` and `Very Negative` groups, which should be considered when interpreting categorical balance.

### Cross-Dataset Integrity

**Finding:** The datasets can be linked through `customer_id`, but customer coverage is incomplete across the three cleaned datasets.

**Evidence:**

- 2,378 customers appear in Customer Demographics.
- 1,642 unique customers appear in Customer Transactions.
- 1,640 unique customers appear in Social Media Interactions.
- 336 transaction customer IDs are not present in the cleaned demographics dataset.
- 336 social-media customer IDs are not present in the cleaned demographics dataset.
- 718 customers are represented in all three cleaned datasets.
- Transaction customer coverage of the cleaned demographics dataset is 54.92%.
- Social-media customer coverage of the cleaned demographics dataset is 54.84%.

**Interpretation:** Cross-dataset analysis is possible, but the datasets should not be treated as having complete one-to-one customer coverage. Analyses that combine demographics with transaction or social-media information must account for unmatched customer IDs and clearly state the population represented after joining the datasets.

### Verification Assessment

Overall, the finalized datasets are complete at the individual-file level, contain unique primary records, and have valid numerical, categorical, and temporal fields. The main limitation identified during verification is incomplete customer alignment across datasets. This does not prevent EDA, but it must be considered when performing cross-dataset comparisons and customer-level analysis.

# 5. Data Cleaning and Preprocessing Review

The datasets used in this milestone are the finalized cleaned datasets produced during the Week 3 preprocessing activity.

The original contaminated datasets were preserved separately, while cleaned analytical copies were created for analysis.

The preprocessing process addressed the major data-quality issues identified in the original datasets, including:

- duplicate records;
- conflicting primary identifiers;
- missing required values;
- invalid or non-numeric age values;
- inconsistent transaction amount values;
- negative transaction amounts;
- inconsistent date formats; and
- categorical formatting inconsistencies.

The objective of preprocessing was not simply to reduce the number of records, but to produce datasets that are consistent, reproducible, and suitable for statistical analysis.

The verification results in Section 4 confirm that the finalized datasets contain no remaining missing values, exact duplicate records, duplicate primary identifiers, or unparseable dates.

The main issue that remains is incomplete customer alignment across the three datasets, which will be treated as an analytical limitation rather than a cleaning error.

In [11]:
# @title
# Final validation of the cleaned datasets before beginning EDA.
# This summarizes the main data-quality checks in one table.

validation_rows = []

for dataset_name, df in datasets.items():

    id_column = id_columns[dataset_name]
    date_column = date_columns[dataset_name]

    validation_rows.append({
        "Dataset": dataset_name,
        "Rows": len(df),
        "Columns": df.shape[1],
        "Missing Values": int(df.isna().sum().sum()),
        "Exact Duplicates": int(df.duplicated().sum()),
        "Duplicate Primary IDs": int(
            df[id_column].duplicated().sum()
        ),
        "Unparseable Dates": int(
            df[date_column].isna().sum()
        )
    })

post_cleaning_validation = pd.DataFrame(
    validation_rows
)

display(post_cleaning_validation)

,Dataset,Rows,Columns,Missing Values,Exact Duplicates,Duplicate Primary IDs,Unparseable Dates
0,Customer Demographics,2378,6,0,0,0,0
1,Customer Transactions,2420,6,0,0,0,0
2,Social Media Interactions,2411,6,0,0,0,0


## 5.1 Cleaning Decision Summary

The preprocessing approach followed the sequence:

**Issue → Treatment → Reason → Validation**

| Issue | Treatment | Reason |
|---|---|---|
| Exact duplicate records | Removed during preprocessing | Prevent repeated records from influencing statistical results |
| Conflicting primary identifiers | Unresolved conflicting records were excluded when a correct value could not be determined | Avoid selecting one conflicting record without supporting evidence |
| Missing required values | Records with unresolved required values were excluded | No reliable basis was available for unsupported imputation |
| Invalid age values | Invalid or non-numeric values were removed and valid ages were stored numerically | Preserve valid quantitative age measurements |
| Inconsistent transaction amounts | Transaction amounts were converted into a consistent numerical format | Enable reliable monetary calculations |
| Negative transaction amounts | Removed when their business meaning could not be verified | No refund or reversal indicator was available to confirm their validity |
| Inconsistent date formats | Parsed and standardized | Enable consistent temporal analysis |
| Categorical formatting | Standardized where necessary | Prevent equivalent categories from being treated as separate values |

The cleaned outputs were then revalidated before EDA. The verification confirmed that the retained records contain no missing values, exact duplicates, duplicate primary identifiers, or unparseable dates.

Cross-dataset customer mismatches were retained and documented because removing unmatched customers solely to force dataset alignment would change the populations represented by the original data.

# 6. Exploratory Data Analysis

With the cleaned datasets verified, the analysis can now focus on describing and understanding the available FinMark data.

The EDA follows the reasoning sequence:

**Business Question → Analytical Method → Result → Interpretation**

The analysis will examine:

- measures of central tendency;
- measures of variability;
- quartiles and potential outliers;
- numerical distributions;
- categorical frequencies and data balance;
- grouped comparisons;
- temporal patterns; and
- relationships across customer demographics, transactions, and social-media activity.

Statistical methods will be selected according to the analytical meaning of each variable rather than applying the same calculations to every column.

Identifiers such as `customer_id`, `transaction_id`, and `interaction_id` will be used for record identification and dataset relationships, not as numerical measurements.

In [12]:
# @title
# Examine the center of the main numerical variables.
# Mean = arithmetic average
# Median = middle value
# Mode = most frequently occurring value

# AGE
age_mean = demographics["age"].mean()
age_median = demographics["age"].median()
age_modes = demographics["age"].mode().tolist()
age_mode_frequency = demographics["age"].value_counts().max()

# TRANSACTION AMOUNT
amount_mean = transactions["amount"].mean()
amount_median = transactions["amount"].median()
amount_modes = transactions["amount"].mode().tolist()
amount_mode_frequency = transactions["amount"].value_counts().max()


print("AGE - CENTRAL TENDENCY")
print("Mean:", fmt_stat(age_mean))
print("Median:", fmt_flexible(age_median))
print(
    "Mode:",
    ", ".join(fmt_whole(value) for value in age_modes[:10])
)
print("Mode Frequency:", fmt_whole(age_mode_frequency))


print("\nTRANSACTION AMOUNT - CENTRAL TENDENCY")
print("Mean:", fmt_amount(amount_mean))
print("Median:", fmt_amount(amount_median))

# Transaction amounts have many unique values, so several values may tie as modes.
if len(amount_modes) <= 10:
    print(
        "Mode:",
        ", ".join(fmt_amount(value) for value in amount_modes)
    )
else:
    print(
        "Mode:",
        ", ".join(fmt_amount(value) for value in amount_modes[:10]),
        f"... ({len(amount_modes)} modal values)"
    )

print("Mode Frequency:", fmt_whole(amount_mode_frequency))

AGE - CENTRAL TENDENCY
Mean: 44.73
Median: 45
Mode: 62
Mode Frequency: 64

TRANSACTION AMOUNT - CENTRAL TENDENCY
Mean: 495.25
Median: 502.12
Mode: 0.00
Mode Frequency: 54


In [13]:
# @title
# Measure how widely age and transaction amounts are distributed.
# Variance and standard deviation are calculated from the cleaned observations.

age_variability = {
    "Minimum": demographics["age"].min(),
    "Maximum": demographics["age"].max(),
    "Range": demographics["age"].max() - demographics["age"].min(),
    "Variance": demographics["age"].var(),
    "Standard Deviation": demographics["age"].std()
}

amount_variability = {
    "Minimum": transactions["amount"].min(),
    "Maximum": transactions["amount"].max(),
    "Range": transactions["amount"].max() - transactions["amount"].min(),
    "Variance": transactions["amount"].var(),
    "Standard Deviation": transactions["amount"].std()
}


print("AGE - VARIABILITY")
print("Minimum:", fmt_whole(age_variability["Minimum"]))
print("Maximum:", fmt_whole(age_variability["Maximum"]))
print("Range:", fmt_whole(age_variability["Range"]))
print("Variance:", fmt_stat(age_variability["Variance"]))
print(
    "Standard Deviation:",
    fmt_stat(age_variability["Standard Deviation"])
)


print("\nTRANSACTION AMOUNT - VARIABILITY")
print("Minimum:", fmt_amount(amount_variability["Minimum"]))
print("Maximum:", fmt_amount(amount_variability["Maximum"]))
print("Range:", fmt_amount(amount_variability["Range"]))
print("Variance:", fmt_stat(amount_variability["Variance"]))
print(
    "Standard Deviation:",
    fmt_amount(amount_variability["Standard Deviation"])
)

AGE - VARIABILITY
Minimum: 18
Maximum: 70
Range: 52
Variance: 238.33
Standard Deviation: 15.44

TRANSACTION AMOUNT - VARIABILITY
Minimum: 0.00
Maximum: 999.86
Range: 999.86
Variance: 87,444.74
Standard Deviation: 295.71


In [14]:
# @title
# Use quartiles and the Interquartile Range (IQR) to examine position,
# spread, and potential statistical outliers.
#
# IQR = Q3 - Q1
#
# Potential outliers are values below:
# Q1 - 1.5 × IQR
#
# or above:
# Q3 + 1.5 × IQR
#
# These observations are flagged for investigation, not automatically removed.

def calculate_iqr(series):

    q1 = series.quantile(0.25)
    median = series.quantile(0.50)
    q3 = series.quantile(0.75)

    iqr = q3 - q1

    lower_bound = q1 - (1.5 * iqr)
    upper_bound = q3 + (1.5 * iqr)

    outlier_mask = (
        (series < lower_bound) |
        (series > upper_bound)
    )

    return {
        "Q1": q1,
        "Median": median,
        "Q3": q3,
        "IQR": iqr,
        "Lower Bound": lower_bound,
        "Upper Bound": upper_bound,
        "Potential Outliers": int(outlier_mask.sum())
    }


age_iqr = calculate_iqr(
    demographics["age"]
)

amount_iqr = calculate_iqr(
    transactions["amount"]
)


print("AGE - QUARTILES AND IQR")
print("Q1:", fmt_flexible(age_iqr["Q1"]))
print("Median:", fmt_flexible(age_iqr["Median"]))
print("Q3:", fmt_flexible(age_iqr["Q3"]))
print("IQR:", fmt_flexible(age_iqr["IQR"]))
print("Lower Bound:", fmt_stat(age_iqr["Lower Bound"]))
print("Upper Bound:", fmt_stat(age_iqr["Upper Bound"]))
print(
    "Potential Outliers:",
    fmt_whole(age_iqr["Potential Outliers"])
)


print("\nTRANSACTION AMOUNT - QUARTILES AND IQR")
print("Q1:", fmt_amount(amount_iqr["Q1"]))
print("Median:", fmt_amount(amount_iqr["Median"]))
print("Q3:", fmt_amount(amount_iqr["Q3"]))
print("IQR:", fmt_amount(amount_iqr["IQR"]))
print("Lower Bound:", fmt_amount(amount_iqr["Lower Bound"]))
print("Upper Bound:", fmt_amount(amount_iqr["Upper Bound"]))
print(
    "Potential Outliers:",
    fmt_whole(amount_iqr["Potential Outliers"])
)

AGE - QUARTILES AND IQR
Q1: 31
Median: 45
Q3: 58
IQR: 27
Lower Bound: -9.50
Upper Bound: 98.50
Potential Outliers: 0

TRANSACTION AMOUNT - QUARTILES AND IQR
Q1: 234.28
Median: 502.12
Q3: 747.75
IQR: 513.48
Lower Bound: -535.94
Upper Bound: 1,517.97
Potential Outliers: 0


## 6.1 Numerical Distribution Analysis

The numerical EDA indicates that the cleaned age and transaction amount variables are suitable for further analysis.

### Customer Age

- Mean age: **44.73 years**
- Median age: **45 years**
- Mode: **62 years**, occurring **64** times
- Minimum and maximum: **18 to 70 years**
- Standard deviation: **15.44 years**
- Q1: **31 years**
- Q3: **58 years**
- IQR: **27 years**
- Potential IQR outliers: **0**

The mean and median are very close, indicating that the center of the age distribution is consistently located around the mid-40s. The IQR shows that the middle 50% of customers are between ages 31 and 58. No age observations fall outside the 1.5 × IQR bounds.

### Transaction Amount

- Mean transaction amount: **495.25**
- Median transaction amount: **502.13**
- Mode: **0.00**, occurring **54** times
- Minimum and maximum: **0.00 to 999.86**
- Standard deviation: **295.71**
- Q1: **234.28**
- Q3: **747.75**
- IQR: **513.48**
- Potential IQR outliers: **0**

The mean and median transaction amounts are also relatively close. Transaction values are widely dispersed across the observed range, but none are flagged as statistical outliers using the 1.5 × IQR rule.

The modal value of **0.00** should be interpreted in the context of the Week 3 preprocessing rule, where `"Free"` transactions were represented as zero together with valid numeric zero values. It therefore should not be treated as the typical monetary transaction amount.

These results describe the numerical distributions statistically. Visual representation of these distributions is deferred to Milestone 2.

In [15]:
# @title
# Summarize the frequency and percentage distribution of categorical variables.

def category_summary(df, column):
    counts = df[column].value_counts(dropna=False)
    percentages = (
        df[column]
        .value_counts(dropna=False, normalize=True)
        .mul(100)
    )

    summary = pd.DataFrame({
        "Count": counts.astype(int),
        "Percentage": percentages
    })

    return summary


categorical_frequency_checks = {
    "Gender": (demographics, "gender"),
    "Income Level": (demographics, "income_level"),
    "Product Category": (transactions, "product_category"),
    "Payment Method": (transactions, "payment_method"),
    "Platform": (social, "platform"),
    "Interaction Type": (social, "interaction_type"),
    "Sentiment": (social, "sentiment")
}

for label, (df, column) in categorical_frequency_checks.items():
    print(f"\n{label.upper()}")

    summary = category_summary(df, column)

    display(
        summary.style.format({
            "Count": "{:,.0f}",
            "Percentage": "{:.2f}%"
        })
    )


GENDER


,Count,Percentage
gender,,
Female,"1,189",50.00%
Male,"1,189",50.00%



INCOME LEVEL


,Count,Percentage
income_level,,
High,810,34.06%
Low,808,33.98%
Medium,760,31.96%



PRODUCT CATEGORY


...

,Count,Percentage
platform,,
Instagram,826,34.26%
Twitter,795,32.97%
Facebook,790,32.77%



INTERACTION TYPE


,Count,Percentage
interaction_type,,
Comment,808,33.51%
Share,808,33.51%
Like,795,32.97%



SENTIMENT


,Count,Percentage
sentiment,,
Neutral,790,32.77%
Positive,790,32.77%
Negative,780,32.35%
Very Positive,26,1.08%
Very Negative,25,1.04%
